what mask lable do i need to use



For a standard KiTS23 (.nii.gz) dataset, the specific mask label you need to use depends entirely on what anatomical structure you want your machine learning model to study.The ground-truth segmentation.nii.gz files in KiTS23 use a hierarchical, multi-class labeling system:

Voxel ValueTarget ROIWhat It Represents

1KidneyThe entire kidney volume (including any internal tumors or cysts).
2TumorMalignant tumor masses only.
3CystBenign fluid-filled kidney cysts.

Which one should you choose?Set MASK_LABEL = 2 (Recommended for Radiomics ML): If your goal is to extract features from the cancer/tumor tissue itself to predict things like pathology, survival rates, or genetic subtypes, you must change your label to 2.Set MASK_LABEL = 1: Only use this if you want to extract features from the overall kidney organ structure rather than isolating the tumor.Where to update it:Simply change the variable at the top of your configuration block before building the extractor:



# Change this to 2 to target the tumor tissue exclusively
MASK_LABEL = 2

In [ ]:
import logging
from pathlib import Path

# ─── PATHS ───────────────────────────────────────────────────────────────────
# Original image directories
SOURCE_DIR = Path("/content/drive/MyDrive/Train-5/")

# CSV output directory
CSV_OUT = Path("/content/drive/MyDrive/CSV/kits23/")
OUTPUT_CSV = CSV_OUT / "train5.csv"

# ─── MASK ────────────────────────────────────────────────────────────────────
# The integer value used by the extractor to isolate your target ROI
MASK_LABEL = 1    

# ─── PYRADIOMICS EXTRACTOR SETTINGS ──────────────────────────────────────────
EXTRACTOR_SETTINGS = {

    # ── Interpolation ────────────────────────────────────────────────────────
    'interpolator'             : 'sitkBSpline',    
    'resampledPixelSpacing'    : [1, 1, 1],         
    'padDistance'              : 10,                

    # ── Mask validation ──────────────────────────────────────────────────────
    'label'                    : MASK_LABEL,        
    'minimumROIDimensions'     : 2,                 
    'minimumROISize'           : 50,                

    # ── Intensity normalisation ───────────────────────────────────────────────
    'binWidth'                 : 5,

    # ── Image pre-processing ─────────────────────────────────────────────────
    'normalize'                : True,   
    'normalizeScale'           : 100,
    'removeOutliers'           : None,    

    # ── Filters (image types) ────────────────────────────────────────────────
    'enabledImageTypes' : {
        'Original' : {},
        'LoG'      : {'sigma': [1.0, 2.0, 3.0]},   
        'Wavelet'  : {},
    },

    # ── Feature classes ───────────────────────────────────────────────────────
    'enabledFeatures' : {
        'shape'         : [],   
        'firstorder'    : [],   
        'glcm'          : [],   
        'glrlm'         : [],   
        'glszm'         : [],   
        'gldm'          : [],   
        'ngtdm'         : [],   
    },

    # ── GLCM-specific ────────────────────────────────────────────────────────
    'weightingNorm'            : None,   

    # ── Verbosity ────────────────────────────────────────────────────────────
    'verbosity'                : logging.WARNING,
}

print("NIfTI Configuration loaded cleanly.")
print(f"  Source : {SOURCE_DIR}")
print(f"  Output : {OUTPUT_CSV}")

In [ ]:
SOURCE_DIR          = Path("/content/drive/MyDrive/RCC-hdf5/extracted_pids-3")
OUTPUT_DIR          = Path("/content/drive/MyDrive/ozyegin/CSV/")
PHASE               = "arterial"
PERTURBATION_FACTOR = 0.30

all_files = sorted([f for f in os.listdir(SOURCE_DIR) if f.endswith(".hdf5")])
print(f"Total HDF5 files found: {len(all_files)}")
print(f"First 5: {all_files[:5]}")

In [ ]:
def build_nifti_extractor(settings: dict) -> featureextractor.RadiomicsFeatureExtractor:
    """
    Construct a PyRadiomics extractor from your plain Python dict.
    Separates top-level params, enabledImageTypes, and enabledFeatures
    to keep configuration reproducible for NIfTI processing.
    """
    # Use .copy() to preserve the core dictionary keys over multiple execution runs
    settings_copy = settings.copy()
    
    enabled_image_types = settings_copy.pop('enabledImageTypes', {'Original': {}})
    enabled_features    = settings_copy.pop('enabledFeatures',  {})

    extractor = featureextractor.RadiomicsFeatureExtractor(**settings_copy)

    # Reset state
    extractor.disableAllImageTypes()
    for img_type, params in enabled_image_types.items():
        extractor.enableImageTypeByName(img_type, customArgs=params if params else None)

    extractor.disableAllFeatures()
    for feature_class, features in enabled_features.items():
        if features:
            extractor.enableFeaturesByName(**{feature_class: features})
        else:
            extractor.enableFeatureClassByName(feature_class)

    return extractor


# Build using your active configuration settings variables
extractor = build_nifti_extractor(radiomics_settings)
print("Extractor ready for NIfTI volumes.")
# FIXED: Capitalized 'T' in enabledImageTypes to prevent native PyRadiomics crash
print(f"  Active image types : {list(extractor.enabledImageTypes.keys())}")
print(f"  Active features    : {list(extractor.enabledFeatures.keys())}")

In [ ]:
from pathlib import Path

# Original image directories
SOURCE_DIR = Path("/content/drive/MyDrive/Train-5/")

# CSV output directory
CSV_OUT = Path("/content/drive/MyDrive/CSV/kits23/")

# Ensure output directory exists
CSV_OUT.mkdir(parents=True, exist_ok=True)

print("Directories initialized:")
print(f"  Source: {SOURCE_DIR}")
print(f"  Output: {CSV_OUT}")

are there any update regarding label kind of things

o. For your NIfTI dataset (which uses the KiTS23 dataset format), the tumor labels are handled differently because they are structurally isolated from your HDF5 setup.

Here is the update regarding how labels work in this NIfTI pipeline:

1. Label Mapping is Discarded
In your previous HDF5 pipeline, we had to worry about string properties like pathology and pathology_grade because the files came from a mixed hospital registry database.

KiTS23 folders do not contain these clinical metadata strings inside the file headers. They are pure image volumes where the region of interest (ROI) is already isolated.

2. The label Parameter is Explicit
In the code block we just created, look at the run_extraction_on_directory arguments:



run_extraction_on_directory(root_dir, extractor, output_csv, label="RCC")


You will pass your label manually as a text string or integer when you call the execution function. This string is automatically injected into every single processed case row as features["label"] = label.


## BURASI COK ONEMLI
3. Mask Integer Voxel Value (MASK_LABEL = 1)
Your extractor uses 1 to locate the tumor pixels inside segmentation.nii.gz. In KiTS23:

0 = Background

1 = Kidney tissue

2 = Tumor mass

3 = Cystic structures

⚠️ Important check: If you intend to extract radiomics features exclusively from the tumor mass in KiTS23, ensure that your MASK_LABEL variable or the label parameter inside your radiomics_settings dictionary is set to 2. If it is set to 1, PyRadiomics will calculate features for the entire healthy kidney volume instead of the tumor!

What does your final execution cell look like where you call run_extraction_on_directory? Paste it whenever you are ready!

In [ ]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk

def extract_features_nifti(case_dir, extractor, label=None):
    """
    Extracts radiomics features and spatial metadata from a single NIfTI case directory.

    Parameters:
        case_dir  : str — path to case folder containing imaging.nii.gz
                    and segmentation.nii.gz
        extractor : RadiomicsFeatureExtractor — initialised with settings
        label     : str or None — optional label to add as a column

    Returns:
        dict of features + metadata, or None if extraction failed
    """
    image_path = os.path.join(case_dir, "imaging.nii.gz")
    mask_path  = os.path.join(case_dir, "segmentation.nii.gz")

    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        print(f"  SKIPPED: missing files in {case_dir}")
        return None

    try:
        # sitk.ReadImage reads real voxel spacing and orientation from NIfTI header
        image = sitk.ReadImage(image_path)
        mask  = sitk.ReadImage(mask_path)

        # ── Extract Spatial Metadata (Matches HDF5 Output) ───────────────────
        ct_spacing   = [round(x, 4) for x in image.GetSpacing()]
        mask_spacing = [round(x, 4) for x in mask.GetSpacing()]
        ct_shape     = list(image.GetSize())
        
        # Count non-zero mask voxels to replicate mask_voxel_count
        mask_array = sitk.GetArrayFromImage(mask)
        mask_voxel_count = int(np.sum(mask_array == 1))

        # ── Execute PyRadiomics ──────────────────────────────────────────────
        features = dict(extractor.execute(image, mask))
        
        # ── Append Identical Metadata Keys ───────────────────────────────────
        features["PID"]             = os.path.basename(case_dir)
        features["case_id"]         = os.path.basename(case_dir)
        features["phase"]           = "arterial" # Default phase for KiTS23 volumes
        features["ct_spacing_mm"]   = str(ct_spacing)
        features["mask_spacing_mm"] = str(mask_spacing)
        features["ct_shape"]        = str(ct_shape)
        features["mask_voxel_count"] = mask_voxel_count
        
        if label is not None:
            features["label"] = label
            
        return features

    except Exception as e:
        print(f"  ERROR in {os.path.basename(case_dir)}: {e}")
        return None


def run_extraction_on_directory(root_dir, extractor, output_csv, label=None):
    """
    Runs feature extraction on all case subdirectories in root_dir.
    Saves results to output_csv.
    """
    cases    = sorted([c for c in os.listdir(root_dir)
                       if os.path.isdir(os.path.join(root_dir, c))])
    data_rows = []

    print(f"\nExtracting from: {root_dir}")
    print(f"Cases found: {len(cases)}")

    for case in cases:
        case_dir = os.path.join(root_dir, case)
        print(f"  Processing: {case}")
        row = extract_features_nifti(case_dir, extractor, label)
        if row is not None:
            data_rows.append(row)

    if data_rows:
        df = pd.DataFrame(data_rows)
        # Ensure target file directory structure exists before writing to disk
        os.makedirs(os.path.dirname(output_csv), exist_ok=True)
        df.to_csv(output_csv, index=False)
        print(f"  Saved {len(df)} rows → {output_csv}")
    else:
        print(f"  No valid cases extracted.")

    return data_rows

In [ ]:
# ─── EXECUTE NIFTI EXTRACTION PIPELINE ────────────────────────────────────────

results = run_extraction_on_directory(
    root_dir   = SOURCE_DIR,
    extractor  = extractor,
    # FIXED: Slashed path notation replaces os.path.join to prevent Pathobject TypeError
    output_csv = CSV_OUT / "train5.csv",
)

PROMPT FOR ABOVE


📊 KiTS23 Structural Verification (Voxel Reminder)Since this script executes extraction loops over raw KiTS23 (Train-5) target cases, it is vital to double-check which specific anatomy region your pipeline is measuring.If you left your radiomics_settings["label"] or MASK_LABEL configuration unassigned, PyRadiomics defaults to reading voxel integer label 1.
Here is exactly what those voxel integer maps represent inside segmentation.nii.gz files:

Voxel ValueTarget Anatomy ROIWhat It Extracts
1Kidney OrgansCalculates texture/shape profiles of the whole kidney.
2Tumor MassCalculates texture/shape profiles of the malignant tumor.
3Kidney CystsCalculates texture/shape profiles of fluid-filled cysts.


If your machine learning objective is to predict tumor properties specifically, verify that your configuration cell is target-locked to use 2 before running this cell block!
Please paste your column reordering and final validation dataframe snippets for the NIfTI pipeline whenever you are ready!

In [ ]:
import pandas as pd

csv_files = sorted([f for f in os.listdir(CSV_OUT) if f.endswith(".csv")])
print(f"{'File':<45} {'Rows':>6} {'Cols':>6}")
print("-" * 60)
for fname in csv_files:
    df = pd.read_csv(os.path.join(CSV_OUT, fname))
    print(f"{fname:<45} {len(df):>6} {len(df.columns):>6}")

# Confirm spacing was applied — diagnostics column should show 1x1x1
print("\n--- Spacing verification (should be [1.0, 1.0, 1.0] after resampling) ---")
sample_csv = os.path.join(CSV_OUT, "kits23_original_aml.csv")
if os.path.exists(sample_csv):
    df = pd.read_csv(sample_csv)
    spacing_col = [c for c in df.columns if "Spacing" in c]
    if spacing_col: